# Clustering inicial de películas (sin PCA)

Este notebook realiza **únicamente el clustering inicial** sobre `movies_2026.csv`.

Objetivo:
- preparar variables numéricas + categóricas simples,
- comparar algoritmos escalables de clustering,
- etiquetar clusters y generar perfiles básicos.

Fuera de alcance (a propósito):
- **PCA** y cualquier otra reducción de dimensionalidad.
- Ingeniería avanzada de texto/multivalor (`genres`, `actors`, etc.).

Esto permite que otro integrante del equipo implemente PCA en una etapa separada.


In [ ]:
# Ejecutar una sola vez por entorno (puede tardar unos minutos).
%pip install pandas numpy scikit-learn matplotlib seaborn jupyter


In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.cluster import KMeans, MiniBatchKMeans, Birch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

RANDOM_STATE = 42


In [ ]:
CSV_PATH = Path('/tmp/codex-lab2-publish.kgl7pm/Lab2/movies_2026.csv')
if not CSV_PATH.exists():
    # Fallback útil si el repo se clona en otra ruta.
    local_candidate = Path.cwd() / 'movies_2026.csv'
    if local_candidate.exists():
        CSV_PATH = local_candidate

OUTPUT_PATH = Path('/tmp/codex-lab2-publish.kgl7pm/Lab2/movies_2026_clusters_initial.csv')
if not OUTPUT_PATH.parent.exists():
    OUTPUT_PATH = Path.cwd() / 'movies_2026_clusters_initial.csv'

REQUIRED_COLUMNS = [
    'id', 'title', 'budget', 'revenue', 'runtime', 'popularity', 'voteAvg', 'voteCount',
    'genresAmount', 'productionCoAmount', 'productionCountriesAmount', 'actorsAmount',
    'castWomenAmount', 'castMenAmount', 'releaseYear', 'originalLanguage', 'video'
]

NUMERIC_COLS = [
    'budget', 'revenue', 'runtime', 'popularity', 'voteAvg', 'voteCount',
    'genresAmount', 'productionCoAmount', 'productionCountriesAmount', 'actorsAmount',
    'castWomenAmount', 'castMenAmount', 'releaseYear'
]

CATEGORICAL_COLS = ['originalLanguage', 'video']
LOG_COLS = ['budget', 'revenue', 'popularity', 'voteCount']


def read_csv_robusto(path: Path) -> pd.DataFrame:
    read_kwargs = {
        'na_values': ['NA', ''],
        'keep_default_na': True,
        'low_memory': False,
    }
    try:
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace', **read_kwargs)
    except TypeError:
        # Compatibilidad con versiones de pandas sin encoding_errors.
        with open(path, 'r', encoding='utf-8', errors='replace', newline='') as f:
            return pd.read_csv(f, **read_kwargs)


def validate_required_columns(df) -> None:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f'Faltan columnas requeridas: {missing}')


def normalize_video(series) -> pd.Series:
    s = series.copy()
    s = s.astype('string').str.strip().str.upper()
    s = s.fillna('UNKNOWN')
    s = s.replace({'': 'UNKNOWN', '<NA>': 'UNKNOWN', 'NA': 'UNKNOWN'})
    return s


def bucket_top_languages(series, top_n=10) -> pd.Series:
    s = series.astype('string').str.strip().str.lower().fillna('unknown')
    s = s.replace({'': 'unknown', 'na': 'unknown', '<na>': 'unknown'})
    top_values = s.value_counts(dropna=False).head(top_n).index
    s = s.where(s.isin(top_values), 'other')
    return s.str.upper()


def build_feature_dataframe(df) -> pd.DataFrame:
    out = df.copy()

    # Conversión numérica robusta.
    for col in NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors='coerce')

    # Normalización de categóricas simples.
    out['video'] = normalize_video(out['video'])
    out['originalLanguage'] = bucket_top_languages(out['originalLanguage'], top_n=10)

    # Evitar problemas con log1p en valores negativos.
    for col in LOG_COLS:
        out[col] = out[col].where(out[col].isna() | (out[col] >= 0), np.nan)
        out[col] = np.log1p(out[col])

    return out


def make_one_hot_encoder():
    # Compatibilidad sklearn: sparse_output (nuevo) vs sparse (antiguo).
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)


def to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else np.asarray(X)


def build_kmeans(n_clusters: int, random_state: int):
    try:
        return KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    except TypeError:
        return KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)


def build_minibatch_kmeans(n_clusters: int, random_state: int):
    try:
        return MiniBatchKMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            n_init='auto',
            batch_size=1024,
        )
    except TypeError:
        return MiniBatchKMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            n_init=10,
            batch_size=1024,
        )


def build_birch(n_clusters: int, random_state: int):
    # random_state no aplica a Birch; se mantiene en la firma para homogeneidad.
    _ = random_state
    return Birch(n_clusters=n_clusters, threshold=0.5, branching_factor=50)


def evaluate_algorithm_grid(X_transformed, configs, k_values, random_state=42) -> pd.DataFrame:
    records = []
    n_samples = X_transformed.shape[0]

    for algorithm_name, cfg in configs.items():
        builder = cfg['builder']

        for k in k_values:
            t0 = time.perf_counter()
            row = {
                'algorithm': algorithm_name,
                'k': int(k),
                'status': 'ok',
                'silhouette_score': np.nan,
                'calinski_harabasz_score': np.nan,
                'davies_bouldin_score': np.nan,
                'inertia': np.nan,
                'runtime_sec': np.nan,
                'n_clusters_efectivos': np.nan,
                'cluster_max_share': np.nan,
                'error': None,
            }
            try:
                model = builder(int(k), random_state)
                if hasattr(model, 'fit_predict'):
                    labels = model.fit_predict(X_transformed)
                else:
                    model.fit(X_transformed)
                    labels = getattr(model, 'labels_', None)
                    if labels is None:
                        raise RuntimeError('El modelo no expuso etiquetas de cluster')

                elapsed = time.perf_counter() - t0
                labels = np.asarray(labels)
                unique_labels, counts = np.unique(labels, return_counts=True)
                n_eff = int(len(unique_labels))
                max_share = float(counts.max() / counts.sum()) if counts.size else np.nan
                row.update({
                    'runtime_sec': elapsed,
                    'n_clusters_efectivos': n_eff,
                    'cluster_max_share': max_share,
                    'inertia': float(getattr(model, 'inertia_', np.nan)) if hasattr(model, 'inertia_') else np.nan,
                })

                if n_eff < 2:
                    row['status'] = 'invalid_clusters'
                    records.append(row)
                    continue

                sil_kwargs = {}
                if n_samples > 5000:
                    sil_kwargs = {'sample_size': 5000, 'random_state': random_state}

                row['silhouette_score'] = float(silhouette_score(X_transformed, labels, **sil_kwargs))
                row['calinski_harabasz_score'] = float(calinski_harabasz_score(X_transformed, labels))
                row['davies_bouldin_score'] = float(davies_bouldin_score(X_transformed, labels))
            except Exception as exc:
                row['status'] = 'error'
                row['runtime_sec'] = time.perf_counter() - t0
                row['error'] = str(exc)

            records.append(row)

    return pd.DataFrame(records)


def select_best_model(results_df) -> dict:
    if results_df.empty:
        raise ValueError('No hay resultados para seleccionar modelo.')

    valid = results_df.copy()
    valid = valid[(valid['status'] == 'ok') & results_df['silhouette_score'].notna()].copy()
    valid = valid[valid['n_clusters_efectivos'] >= 2]

    if valid.empty:
        debug_cols = ['algorithm', 'k', 'status', 'error']
        raise ValueError('No hubo combinaciones válidas. Revisar resultados.\n' + str(results_df[debug_cols].head(20)))

    non_degenerate = valid[valid['cluster_max_share'] <= 0.95].copy()
    candidate_pool = non_degenerate if not non_degenerate.empty else valid

    candidate_pool = candidate_pool.sort_values(
        by=['silhouette_score', 'calinski_harabasz_score', 'davies_bouldin_score'],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    best = candidate_pool.iloc[0].to_dict()
    best['degenerate_filter_applied'] = bool(not non_degenerate.empty)
    return best


def profile_clusters(df_labels, numeric_cols, categorical_cols) -> dict:
    output = {}
    output['numeric_mean'] = df_labels.groupby('cluster_label')[numeric_cols].mean(numeric_only=True)
    output['numeric_median'] = df_labels.groupby('cluster_label')[numeric_cols].median(numeric_only=True)

    categorical_profiles = {}
    for col in categorical_cols:
        freq = pd.crosstab(df_labels['cluster_label'], df_labels[col], normalize='index')
        categorical_profiles[col] = freq.sort_index(axis=1)

    output['categorical'] = categorical_profiles
    return output


In [ ]:
print(f'Leyendo dataset desde: {CSV_PATH}')
df_raw = read_csv_robusto(CSV_PATH)
print('Shape inicial:', df_raw.shape)
validate_required_columns(df_raw)

missing_summary = df_raw[REQUIRED_COLUMNS].isna().sum().sort_values(ascending=False)
print('\\nFaltantes (columnas requeridas):')
display(missing_summary.to_frame('missing_count'))

dup_mask = df_raw['id'].duplicated(keep='first')
dup_count = int(dup_mask.sum())
print(f'\\nDuplicados por id detectados: {dup_count}')
if dup_count > 0:
    df_raw = df_raw.loc[~dup_mask].copy()
    print('Shape tras eliminar duplicados (manteniendo la primera ocurrencia):', df_raw.shape)


In [ ]:
# Construcción del dataset de trabajo para clustering.
df_model = build_feature_dataframe(df_raw)

feature_cols = NUMERIC_COLS + CATEGORICAL_COLS
df_features = df_model[['id', 'title'] + feature_cols].copy()

print('Vista previa de features preparadas:')
display(df_features.head())
print('\\nResumen rápido de dtypes:')
display(df_features[feature_cols].dtypes.to_frame('dtype'))


In [ ]:
# Preprocesamiento reproducible (numéricas + categóricas simples).
# FunctionTransformer se importa para mantener compatibilidad con el plan; aquí se usa como paso explícito de identidad.
identity_transformer = FunctionTransformer(lambda x: x, validate=False)

numeric_pipeline = Pipeline(steps=[
    ('identity', identity_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
    ('onehot', make_one_hot_encoder()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, NUMERIC_COLS),
        ('cat', categorical_pipeline, CATEGORICAL_COLS),
    ],
    remainder='drop',
)

X_input = df_features[feature_cols].copy()
X_transformed = preprocessor.fit_transform(X_input)
X_model = to_dense(X_transformed)

print('Forma de X preprocesada (densa para métricas/modelos):', X_model.shape)


In [ ]:
algorithm_configs = {
    'KMeans': {'builder': build_kmeans},
    'MiniBatchKMeans': {'builder': build_minibatch_kmeans},
    'Birch': {'builder': build_birch},
}

k_values = list(range(2, 11))

results_df = evaluate_algorithm_grid(
    X_transformed=X_model,
    configs=algorithm_configs,
    k_values=k_values,
    random_state=RANDOM_STATE,
)

print('Resultados del grid (top 15 por silhouette):')
display(
    results_df.sort_values(
        by=['silhouette_score', 'calinski_harabasz_score', 'davies_bouldin_score'],
        ascending=[False, False, True]
    ).head(15)
)

print('\\nResumen por estado:')
display(results_df['status'].value_counts(dropna=False).to_frame('count'))


In [ ]:
best_selection = select_best_model(results_df)
print('Mejor configuración seleccionada:')
display(pd.Series(best_selection))

best_algorithm = best_selection['algorithm']
best_k = int(best_selection['k'])

best_model = algorithm_configs[best_algorithm]['builder'](best_k, RANDOM_STATE)
if hasattr(best_model, 'fit_predict'):
    final_labels = best_model.fit_predict(X_model)
else:
    best_model.fit(X_model)
    final_labels = best_model.labels_

final_labels = np.asarray(final_labels, dtype=int)

df_clustered = df_model.copy()
df_clustered['cluster_label'] = final_labels

df_output = df_clustered[['id', 'title', 'cluster_label']].copy()
df_output['best_algorithm'] = best_algorithm
df_output['best_k'] = best_k

print('Vista previa de salida etiquetada:')
display(df_output.head())


In [ ]:
# Curva de codo (inertia) para KMeans y MiniBatchKMeans.
elbow_df = results_df[
    (results_df['algorithm'].isin(['KMeans', 'MiniBatchKMeans'])) &
    (results_df['status'] == 'ok') &
    (results_df['inertia'].notna())
].copy()

plt.figure(figsize=(10, 5))
for alg, part in elbow_df.groupby('algorithm'):
    part = part.sort_values('k')
    plt.plot(part['k'], part['inertia'], marker='o', label=alg)
plt.title('Curva de codo (inertia)')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.legend()
plt.show()


In [ ]:
# Métricas internas vs k por algoritmo.
metrics_to_plot = [
    ('silhouette_score', 'Silhouette (mayor es mejor)'),
    ('calinski_harabasz_score', 'Calinski-Harabasz (mayor es mejor)'),
    ('davies_bouldin_score', 'Davies-Bouldin (menor es mejor)'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_df = results_df[results_df['status'] == 'ok'].copy()

for ax, (metric_col, title) in zip(axes, metrics_to_plot):
    for alg, part in plot_df.groupby('algorithm'):
        part = part.sort_values('k')
        ax.plot(part['k'], part[metric_col], marker='o', label=alg)
    ax.set_title(title)
    ax.set_xlabel('k')
    ax.set_ylabel(metric_col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Distribución de tamaños del modelo ganador.
cluster_size = df_clustered['cluster_label'].value_counts().sort_index()
cluster_share = (cluster_size / cluster_size.sum()).rename('share')

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=cluster_size.index.astype(str), y=cluster_size.values, ax=ax, palette='viridis')
ax.set_title(f'Tamaño de clusters ({best_algorithm}, k={best_k})')
ax.set_xlabel('cluster_label')
ax.set_ylabel('n películas')
plt.show()

print('Tamaños de cluster:')
display(pd.DataFrame({'count': cluster_size, 'share': cluster_share}).sort_index())


In [ ]:
profiles = profile_clusters(df_clustered, NUMERIC_COLS, CATEGORICAL_COLS)

numeric_profile = profiles['numeric_median'].copy()
if numeric_profile.shape[0] >= 2:
    # Estandarización por columna para hacer comparables magnitudes en el heatmap.
    denom = numeric_profile.std(axis=0).replace(0, np.nan)
    numeric_profile_z = (numeric_profile - numeric_profile.mean(axis=0)) / denom
else:
    numeric_profile_z = numeric_profile.copy()

plt.figure(figsize=(14, 6))
sns.heatmap(numeric_profile_z, cmap='coolwarm', center=0, annot=False)
plt.title('Heatmap de perfiles numéricos por cluster (medianas estandarizadas)')
plt.xlabel('Variables numéricas')
plt.ylabel('cluster_label')
plt.show()

print('Perfiles numéricos (medianas):')
display(profiles['numeric_median'])

for cat_col, cat_table in profiles['categorical'].items():
    print(f'\\nDistribución relativa por cluster - {cat_col}')
    display(cat_table)


In [ ]:
# Exportación para handoff al equipo (activo por defecto).
df_output.to_csv(OUTPUT_PATH, index=False)
print(f'Archivo exportado: {OUTPUT_PATH.resolve()}')
print(f'Filas exportadas: {len(df_output):,}')


## Handoff para el equipo (PCA en una etapa separada)

Este notebook deja lista la etapa de clustering inicial y la salida etiquetada (`cluster_label`).

Siguiente paso sugerido para otro integrante:
- aplicar PCA (u otra reducción) sobre las **mismas features preprocesadas** o una variante acordada,
- comparar visualizaciones/proyecciones con las etiquetas obtenidas aquí,
- documentar diferencias sin modificar este notebook base de clustering.

Este notebook **no implementa PCA**, por diseño.
